In [ ]:

from datetime import datetime
from pyspark.sql.types import *
import uuid

# === CONFIGURATION - Change for each notebook ===
NOTEBOOK_NAME = "ntk_silver_Site"        # ← Change this for each notebook
PIPELINE_NAME = "pipeline_test"     # ← Change this for each pipeline
ACTIVITY_TYPE = "DataTransformation"     # ← DataExtract/DataTransform/DataLoad/DataValidation
SOURCE_PATH = "abfss://Bronze/Sites_Inventory" # ← Change source path
TARGET_PATH = "abfss://silver/Dim_Site" # ← Change target path

def log_etl_activity(status, start_time=None, error=None, **metrics):
    """Log ETL activity to pipeline table"""
    current_time = datetime.now()
    
    # Get next LogID
    try:
        log_id = spark.sql("SELECT COALESCE(MAX(LogID), 0) + 1 as id FROM etl_silver_pipeline_log").collect()[0]['id']
    except:
        log_id = 1
    
    if status == "STARTED":
        data = [(
            log_id, PIPELINE_NAME, f"run_{current_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            current_time, None, None, "RUNNING", None, 
            SOURCE_PATH, TARGET_PATH, None, None, None, None, 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
        start_time = current_time
        
    else:
        duration = int((current_time - start_time).total_seconds()) if start_time else None
        data = [(
            log_id, PIPELINE_NAME, f"run_{start_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            start_time, current_time, duration, status, 
            str(error) if error else None, SOURCE_PATH, TARGET_PATH,
            metrics.get('rows_read'), metrics.get('rows_written'), 
            metrics.get('file_count'), metrics.get('bytes_processed'), 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
    
    # Schema for etl_silver_pipeline_log table
    schema = StructType([
        StructField("LogID", LongType()), StructField("PipelineName", StringType()),
        StructField("RunID", StringType()), StructField("ActivityName", StringType()),
        StructField("ActivityType", StringType()), StructField("NotebookName", StringType()),
        StructField("Sequence", IntegerType()), StructField("StartTime", TimestampType()),
        StructField("EndTime", TimestampType()), StructField("DurationSeconds", IntegerType()),
        StructField("Status", StringType()), StructField("ErrorMessage", StringType()),
        StructField("SourcePath", StringType()), StructField("TargetPath", StringType()),
        StructField("RowsRead", LongType()), StructField("RowsWritten", LongType()),
        StructField("FileCountProcessed", IntegerType()), StructField("BytesProcessed", LongType()),
        StructField("InsertedOn", TimestampType()), StructField("InsertedBy", StringType()),
        StructField("CorrelationID", StringType())
    ])
    
    # Save to table
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable("etl_silver_pipeline_log")
    
    # Print status
    if status == "STARTED":
        print(f"🚀 Starting {NOTEBOOK_NAME}")
    elif status == "SUCCESS":
        duration_text = f" ({duration}s)" if duration else ""
        print(f"✅ {NOTEBOOK_NAME} completed successfully{duration_text}")
    else:
        print(f"❌ {NOTEBOOK_NAME} failed")
    
    return current_time if status == "STARTED" else None

# Start logging
print(f"🔧 Initializing {NOTEBOOK_NAME}...")
start_time = log_etl_activity("STARTED")

# Initialize variables for tracking metrics
rows_read = 0
rows_written = 0
file_count = 0
bytes_processed = 0

# print(f"📊 Ready to process data from: {SOURCE_PATH}")
# print(f"🎯 Target location: {TARGET_PATH}")

StatementMeta(, f30d875b-23cb-4fff-a782-8b91d04902c9, 3, Finished, Available, Finished)

🔧 Initializing ntk_silver_Site...
🚀 Starting ntk_silver_Site


## Paths for file reading and saving 

In [ ]:
source_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Bronze_lakehouse.Lakehouse/Files/Bronze_layer/SharePointFiles"
target_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting"

StatementMeta(, f30d875b-23cb-4fff-a782-8b91d04902c9, 4, Finished, Available, Finished)

## Reading file from bronze

In [ ]:
### Reading Site file from bronze then doing some tranformations then load into silver layer as parquet file
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, lit, current_timestamp, to_timestamp, when, upper, regexp_replace, coalesce
)
from pyspark.sql.types import StringType, IntegerType
from datetime import datetime

# Step 1: Spark Session
spark = SparkSession.builder.appName("BronzeToSilver_Sites").getOrCreate()

# Step 2: Paths
today = datetime.now()  
from datetime import datetime, timedelta
today = today - timedelta(days=1)

year = today.strftime("%Y")
month = today.strftime("%m")
day = today.strftime("%d") 
bronze_path = f"{source_path}/{year}/{month}/{day}/Sites_Inventory.csv"

current_date = datetime.now()
year = str(current_date.year)
month = f"{current_date.month:02d}"
day = f"{current_date.day:02d}"

silver_path = f"{target_path}/{year}/{month}/{day}/Dim_Site.parquet"

# Step 3: Read CSV from Bronze

df_raw = spark.read.option("header", True).csv(bronze_path)
df_raw.count()
df_raw.show(1)


StatementMeta(, f30d875b-23cb-4fff-a782-8b91d04902c9, 5, Finished, Available, Finished)

+-------------+--------------------+--------------------+-------------------+------------+-----------+-------------+---------+--------------------+--------------------+-------------------+----------+-------------+-------------+--------------+------------------+-------------+-------------+---------+----------+---------+--------+--------+--------+--------------------+--------------------+------------------+---------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------+-------------------+-------+----------------+----------+--------------------+--------------------+----------------+--------------+-------------+------------------+---------------+-----------------+--------------------+----------------+----------------+----------------+--------------+------------------+-------------------+-------------+
|        Title|                

In [ ]:

# Trim all string columns
string_cols = [c for c, t in df_raw.dtypes if t == "string"]
for col_name in string_cols:
    df_raw = df_raw.withColumn(col_name, trim(col(col_name)))

# Replace empty strings and "null" strings with None
df_cleaned = df_raw.replace("", None).replace("null", None)

## 		OwnerKey
# Drop rows with missing UserID if present
if "SiteID" in df_cleaned.columns:
    df_cleaned = df_cleaned.dropna(subset=["SiteID"])

# Remove duplicates based on if present
if "SiteID" in df_cleaned.columns:
    df_cleaned = df_cleaned.dropDuplicates(["SiteID"])

# Add required columns
if "Tenant_Id" not in df_cleaned.columns:
    df_cleaned = df_cleaned.withColumn("Tenant_Id", lit(None).cast("int"))

if "SensitivityLabel" not in df_cleaned.columns:
    df_cleaned = df_cleaned.withColumn("SensitivityLabel", lit(None).cast("string"))

if "OwnerKey" not in df_cleaned.columns:
    df_cleaned = df_cleaned.withColumn("OwnerKey", lit(None).cast("int"))

# Add SnapshotDate
df_cleaned = df_cleaned.withColumn("SnapShotDate", current_timestamp())

# Add GUID Column
df_cleaned = df_cleaned.withColumn("Site_GUID",  col("Id"))

df_cleaned = df_cleaned.withColumnRenamed("url", "Site_Url")

# Show the first 1 rows and count
df_cleaned.show(1)
df_cleaned.count()


StatementMeta(, f30d875b-23cb-4fff-a782-8b91d04902c9, 6, Finished, Available, Finished)

+-------------+--------------------+--------------------+-------------------+------------+-----------+-------------+---------+--------------------+--------------------+-------------------+----------+-------------+-------------+--------------+------------------+-------------+-------------+---------+----------+---------+--------+--------+--------+--------------------+--------------------+------------------+---------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------+-------------------+-------+----------------+----------+--------------------+--------------------+----------------+--------------+-------------+------------------+---------------+-----------------+--------------------+----------------+----------------+----------------+--------------+------------------+-------------------+-------------+---------+--------+-------------

1

In [ ]:
from pyspark.sql.functions import col, when, trim, sha2, coalesce, concat, lit
# Creating SiteKey using Hash Value
df_cleaned = df_cleaned.withColumn(
    "SiteKey",
    when(
        col("Site_Url").isNotNull(),
        sha2(col("Site_Url"), 256)
    ).otherwise(None)
)
df_cleaned.show(1)
df_cleaned.count()


StatementMeta(, f30d875b-23cb-4fff-a782-8b91d04902c9, 7, Finished, Available, Finished)

+-------------+--------------------+--------------------+-------------------+------------+-----------+-------------+---------+--------------------+--------------------+-------------------+----------+-------------+-------------+--------------+------------------+-------------+-------------+---------+----------+---------+--------+--------+--------+--------------------+--------------------+------------------+---------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------+-------------------+-------+----------------+----------+--------------------+--------------------+----------------+--------------+-------------+------------------+---------------+-----------------+--------------------+----------------+----------------+----------------+--------------+------------------+-------------------+-------------+---------+--------+-------------

1

In [ ]:
# # Step 7: Write to Silver as Parquet

df_cleaned.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(silver_path)

print("Dim_Site Parquet file created successfully in Silver layer!")


StatementMeta(, f30d875b-23cb-4fff-a782-8b91d04902c9, 8, Finished, Available, Finished)

Dim_Site Parquet file created successfully in Silver layer!


In [ ]:

# Ensure processing_successful is defined before this block
try:
    print(f"🔄 Starting ETL processing for {NOTEBOOK_NAME}...")

    # Your ETL logic here
    # Example:
    # source_df = spark.read.format("delta").load(SOURCE_PATH)
    # transformed_df = source_df.filter("status = 'active'")
    # transformed_df.write.format("delta").mode("overwrite").save(TARGET_PATH)

    # Simulated metrics
    rows_read = df_raw.count()
    rows_written = df_cleaned.count()
    # file_count = len(dbutils.fs.ls(SOURCE_PATH))
    bytes_processed = 524288000  # ~500MB

    processing_successful = True

except Exception as e:
    error_details = e
    processing_successful = False

# Complete the logging based on processing results
if processing_successful:
    # Log successful completion with metrics
    log_etl_activity("SUCCESS", start_time, 
                     rows_read=rows_read, 
                     rows_written=rows_written,
                    #  file_count=file_count,
                     bytes_processed=bytes_processed)
    
    print(f"🎉 {NOTEBOOK_NAME} pipeline completed successfully!")
    print(f"📊 Final metrics:")
    print(f"   ✅ Status: SUCCESS")
    print(f"   📖 Total rows processed: {rows_read:,} → {rows_written:,}")
    print(f"   🔄 Data throughput: {bytes_processed/(1024**2):.1f} MB")
    
    # Optional: Show recent logs for this notebook
    print(f"\n📋 Recent runs for {NOTEBOOK_NAME}:")
    spark.sql(f"""
        SELECT LogID, Status, StartTime, EndTime, DurationSeconds, RowsRead, RowsWritten
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=True)
    
else:
    # Log failure
    log_etl_activity("FAILED", start_time, error_details)
    
    print(f"💥 {NOTEBOOK_NAME} pipeline failed!")
    print(f"❌ Error: {str(error_details)}")
    
    # Optional: Show error analysis
    print(f"\n🔍 Recent failures for debugging:")
    spark.sql(f"""
        SELECT LogID, StartTime, ErrorMessage, DurationSeconds
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}' AND Status = 'FAILED'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=False)
    
    # Re-raise the exception to fail the notebook
    raise error_details

# Cleanup variables
print(f"\n🧹 Cleaning up variables...")
del rows_read, rows_written, bytes_processed

print(f"✨ {NOTEBOOK_NAME} logging completed!")


StatementMeta(, f30d875b-23cb-4fff-a782-8b91d04902c9, 9, Finished, Available, Finished)

🔄 Starting ETL processing for ntk_silver_Site...
✅ ntk_silver_Site completed successfully (111s)
🎉 ntk_silver_Site pipeline completed successfully!
📊 Final metrics:
   ✅ Status: SUCCESS
   📖 Total rows processed: 1 → 1
   🔄 Data throughput: 500.0 MB

📋 Recent runs for ntk_silver_Site:
+-----+-------+--------------------+--------------------+---------------+--------+-----------+
|LogID| Status|           StartTime|             EndTime|DurationSeconds|RowsRead|RowsWritten|
+-----+-------+--------------------+--------------------+---------------+--------+-----------+
|  117|RUNNING|2025-10-15 05:31:...|                NULL|           NULL|    NULL|       NULL|
|  118|SUCCESS|2025-10-15 05:31:...|2025-10-15 05:33:...|            111|       1|          1|
|  106|RUNNING|2025-10-14 14:29:...|                NULL|           NULL|    NULL|       NULL|
+-----+-------+--------------------+--------------------+---------------+--------+-----------+


🧹 Cleaning up variables...
✨ ntk_silver_Site lo